# Bài 1: Giới Thiệu về LightGBM và Gradient Boosting

## Mục tiêu bài học

Sau bài học này, bạn sẽ có thể:
- Hiểu được khái niệm cốt lõi của thuật toán Gradient Boosting.
- Giải thích được tại sao LightGBM ra đời và những ưu điểm vượt trội của nó so với các thuật toán Gradient Boosting khác (như XGBoost).
- Nắm được kiến trúc và các thành phần chính của LightGBM: Leaf-wise tree growth, Gradient-based One-Side Sampling (GOSS), và Exclusive Feature Bundling (EFB).
- Cài đặt và huấn luyện một mô hình LightGBM đơn giản cho bài toán classification.

## Tại sao chủ đề này quan trọng?

Trong các cuộc thi Data Science trên Kaggle và trong các ứng dụng thực tế, các mô hình dựa trên Gradient Boosting Decision Tree (GBDT) luôn chiếm ưu thế nhờ hiệu suất dự đoán vượt trội. LightGBM là một trong những framework GBDT hàng đầu, được phát triển bởi Microsoft, nổi tiếng với tốc độ huấn luyện cực nhanh và sử dụng bộ nhớ hiệu quả mà vẫn giữ được độ chính xác cao.

Hiểu và sử dụng thành thạo LightGBM không chỉ giúp bạn xây dựng các mô hình mạnh mẽ hơn mà còn tiết kiệm đáng kể thời gian và tài nguyên tính toán, một yếu tố cực kỳ quan trọng trong các dự án dữ liệu lớn.

## Giải thích trực quan: Gradient Boosting hoạt động như thế nào?

Hãy tưởng tượng bạn đang cố gắng dự đoán giá của một ngôi nhà. Bạn bắt đầu với một dự đoán rất đơn giản, ví dụ như giá trung bình của tất cả các ngôi nhà trong khu vực. Dĩ nhiên, dự đoán này sẽ có sai số (residual) lớn.

**Gradient Boosting hoạt động theo một chuỗi các bước lặp đi lặp lại:**

1.  **Bắt đầu với một mô hình yếu (weak learner):** Thường là một cây quyết định (decision tree) rất nông. Mô hình này đưa ra một dự đoán ban đầu.
2.  **Tính toán sai số (Residuals):** Tính sự khác biệt giữa dự đoán của mô hình và giá trị thực tế. Sai số này cho biết mô hình đã sai ở đâu và sai bao nhiêu.
3.  **Huấn luyện một mô hình mới để dự đoán sai số:** Ở bước này, thay vì dự đoán giá nhà, chúng ta xây dựng một cây quyết định mới để học cách dự đoán chính những sai số từ bước trước. Mục tiêu là để mô hình sau sửa lỗi cho mô hình trước.
4.  **Cập nhật dự đoán:** Cộng dự đoán của mô hình mới (dự đoán sai số) vào dự đoán của mô hình cũ. Điều này giúp "sửa chữa" dần dần dự đoán tổng thể, làm nó tiến gần hơn đến giá trị thực.
5.  **Lặp lại:** Quá trình này được lặp lại nhiều lần. Mỗi mô hình mới được thêm vào đều tập trung vào việc sửa những sai sót còn lại của chuỗi mô hình trước đó. Cuối cùng, mô hình cuối cùng là tổng hợp của tất cả các mô hình yếu đã được huấn luyện.

Bằng cách này, Gradient Boosting xây dựng một mô hình mạnh mẽ từ việc kết hợp nhiều mô hình yếu, mỗi mô hình sau học hỏi từ lỗi của mô hình trước.

## LightGBM: Sự cải tiến vượt trội

Các framework Gradient Boosting truyền thống như XGBoost xây dựng cây theo **Level-wise (hoặc Depth-wise)**. Tức là nó phát triển cây theo từng tầng, đảm bảo cây luôn cân bằng. Cách này dễ lập trình nhưng không hiệu quả, vì nó xử lý các nhánh lá có cùng độ sâu một cách như nhau, kể cả những nhánh lá có loss thấp và không cần thiết phải phân chia thêm.

LightGBM sử dụng một chiến lược thông minh hơn gọi là **Leaf-wise tree growth**.

### 1. Leaf-wise Tree Growth

Thay vì phát triển đồng đều, LightGBM sẽ chọn nhánh lá (leaf) có khả năng giảm loss nhiều nhất và phân chia nó. Điều này giúp mô hình hội tụ nhanh hơn và tạo ra các cây phức tạp hơn ở những nơi cần thiết, dẫn đến độ chính xác cao hơn. Tuy nhiên, nó có thể dẫn đến overfitting nếu không được kiểm soát bằng các tham số như `max_depth`.

![Leaf-wise vs Level-wise](https://raw.githubusercontent.com/microsoft/LightGBM/master/docs/images/leaf-wise.png)
*Nguồn: Microsoft LightGBM Documentation*

### 2. Gradient-based One-Side Sampling (GOSS)

Để tăng tốc độ, LightGBM không sử dụng tất cả các mẫu dữ liệu để tính toán information gain khi phân chia cây. GOSS giữ lại tất cả các mẫu có gradient lớn (là những mẫu bị mô hình dự đoán sai nhiều) và chỉ lấy một mẫu ngẫu nhiên từ các mẫu có gradient nhỏ.

Ý tưởng là các mẫu bị dự đoán sai nhiều chứa nhiều thông tin hơn để cải thiện mô hình. Bằng cách này, GOSS giúp giảm đáng kể lượng dữ liệu cần tính toán mà không làm ảnh hưởng nhiều đến độ chính xác.

### 3. Exclusive Feature Bundling (EFB)

Trong các tập dữ liệu có số chiều lớn (nhiều features), ma trận dữ liệu thường rất thưa (sparse), nghĩa là có nhiều features không bao giờ có giá trị khác 0 cùng một lúc (mutually exclusive). Ví dụ, trong one-hot encoding, các cột mới được tạo ra luôn loại trừ lẫn nhau.

EFB tìm cách gộp các features loại trừ lẫn nhau này thành một feature duy nhất (feature bundle). Điều này giúp giảm số lượng features một cách an toàn, từ đó giảm chi phí tính toán khi tìm điểm chia tốt nhất cho cây.

**Nhờ 3 kỹ thuật này, LightGBM đạt được tốc độ huấn luyện nhanh hơn và sử dụng bộ nhớ ít hơn đáng kể so với các đối thủ.**

## Ví dụ thực tế: Xây dựng mô hình phân loại khách hàng

Bây giờ, chúng ta sẽ áp dụng LightGBM vào một bài toán thực tế: dự đoán liệu một khách hàng có rời bỏ dịch vụ hay không (customer churn) dựa trên bộ dữ liệu `churn` nổi tiếng.

In [ ]:
# Bước 1: Cài đặt và import các thư viện cần thiết
!pip install lightgbm pandas scikit-learn

import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

**Lưu ý:** Nếu bạn đang dùng Google Colab hoặc môi trường không có sẵn dữ liệu, bạn cần tải file `WA_Fn-UseC_-Telco-Customer-Churn.csv` về. Trong ví dụ này, chúng ta sẽ tải trực tiếp từ một URL.

In [ ]:
# Bước 2: Tải và chuẩn bị dữ liệu
url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
df = pd.read_csv(url)

# Xem qua dữ liệu
print("5 dòng đầu của dữ liệu:")
print(df.head())
print("\nThông tin chung về dữ liệu:")
df.info()

Dữ liệu có một số cột dạng `object` (text) cần được chuyển đổi sang dạng số để mô hình có thể học được. Cột `TotalCharges` cũng đang bị đọc sai định dạng và có giá trị rỗng.

In [ ]:
# Bước 3: Tiền xử lý dữ liệu (Data Preprocessing)

# Chuyển TotalCharges sang dạng số, các giá trị lỗi sẽ thành NaN
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Xóa các dòng có giá trị thiếu (NaN)
df.dropna(inplace=True)

# Xóa cột customerID vì không cần thiết cho mô hình
df.drop(columns=['customerID'], inplace=True)

# Sử dụng LabelEncoder để chuyển đổi các cột categorical
label_encoders = {}
for column in df.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    df[column] = le.fit_transform(df[column])
    label_encoders[column] = le

print("\nDữ liệu sau khi mã hóa:")
print(df.head())

Bây giờ tất cả các cột đã ở dạng số, chúng ta có thể bắt đầu huấn luyện mô hình.

In [ ]:
# Bước 4: Phân chia dữ liệu và huấn luyện mô hình LightGBM

# Tách features (X) và target (y)
X = df.drop('Churn', axis=1)
y = df['Churn']

# Phân chia tập train và test (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Khởi tạo mô hình LightGBM Classifier
# Sử dụng các tham số mặc định để bắt đầu
lgb_clf = lgb.LGBMClassifier(random_state=42)

# Huấn luyện mô hình
print("\nBắt đầu huấn luyện mô hình...")
lgb_clf.fit(X_train, y_train)
print("Huấn luyện hoàn tất!")

In [ ]:
# Bước 5: Đánh giá mô hình

# Dự đoán trên tập test
y_pred = lgb_clf.predict(X_test)

# Tính toán độ chính xác
accuracy = accuracy_score(y_test, y_pred)
print(f"\nĐộ chính xác (Accuracy) trên tập test: {accuracy:.4f}")

# Hiển thị confusion matrix
print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(cm)

# Hiển thị classification report chi tiết
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

## Lỗi thường gặp và những quan niệm sai lầm

1.  **"LightGBM luôn tốt hơn XGBoost"**: Không hẳn. Mặc dù LightGBM thường nhanh hơn, nhưng trên các tập dữ liệu nhỏ (dưới 10,000 dòng), XGBoost đôi khi cho kết quả tốt hơn và ít bị overfitting hơn do cách phát triển cây (level-wise) an toàn hơn.
2.  **Quên xử lý dữ liệu Categorical**: LightGBM có khả năng xử lý trực tiếp các cột categorical mà không cần one-hot encoding, giúp tăng hiệu suất. Tuy nhiên, bạn cần chỉ định rõ các cột này cho mô hình bằng cách chuyển kiểu dữ liệu của chúng sang `category` trong Pandas trước khi `fit`. Nếu không, LightGBM sẽ coi chúng như các biến số thông thường, dẫn đến kết quả không tối ưu. (Trong ví dụ trên, chúng ta dùng LabelEncoder, một cách tiếp cận khác).
3.  **Sử dụng tham số mặc định cho mọi bài toán**: Các tham số mặc định của LightGBM được thiết kế để hoạt động tốt trên nhiều loại dữ liệu, nhưng để đạt hiệu suất cao nhất, việc tinh chỉnh (tuning) các tham số như `n_estimators`, `learning_rate`, `num_leaves`, `max_depth` là cực kỳ quan trọng. Chúng ta sẽ tìm hiểu kỹ hơn trong các bài học sau.

## Tóm tắt

- **Gradient Boosting** là một kỹ thuật ensemble mạnh mẽ, xây dựng mô hình bằng cách kết hợp tuần tự nhiều mô hình yếu, mỗi mô hình sau sửa lỗi cho mô hình trước.
- **LightGBM** là một framework Gradient Boosting hiệu suất cao, sử dụng các kỹ thuật **Leaf-wise growth**, **GOSS**, và **EFB** để tăng tốc độ huấn luyện và giảm bộ nhớ sử dụng.
- Bạn đã học cách cài đặt, chuẩn bị dữ liệu và huấn luyện một mô hình LightGBM cơ bản cho bài toán phân loại.

## Bài học tiếp theo

Trong bài học tiếp theo, chúng ta sẽ đi sâu vào các tham số quan trọng nhất của LightGBM và cách chúng ảnh hưởng đến hiệu suất và tốc độ của mô hình. Việc hiểu rõ các tham số này là chìa khóa để bạn có thể "mở khóa" toàn bộ sức mạnh của LightGBM.